# 08 - Model: TF-IDF

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd
from IPython.display import display
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"

tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
print(tracks.shape)

(89740, 40)


## Text descriptor

In [2]:
descriptor_columns = ["track_genre", "mood_label", "tempo_group", "energy_level", "danceability_level"]
descriptor_frame = tracks[descriptor_columns].astype("object").fillna("unknown").astype(str)
tracks["genre_descriptor"] = descriptor_frame.agg(" ".join, axis=1).str.lower()

tracks[["track_genre", "mood_label", "genre_descriptor"]].head()

,track_genre,mood_label,genre_descriptor
0,acoustic,calm_positive,acoustic calm_positive slow medium_energy high...
1,acoustic,calm_dark,acoustic calm_dark slow low_energy medium_dance
2,acoustic,calm_dark,acoustic calm_dark slow medium_energy medium_d...
3,acoustic,calm_dark,acoustic calm_dark very_fast low_energy low_dance
4,acoustic,calm_dark,acoustic calm_dark medium medium_energy medium...


## Fit TF-IDF

In [3]:
vectorizer = TfidfVectorizer()
descriptor_matrix = vectorizer.fit_transform(tracks["genre_descriptor"])
print(descriptor_matrix.shape, len(vectorizer.vocabulary_))

(89740, 129) 129


## Recommend

In [4]:
def recommend_by_genre_query(query: str, top_n: int = 10, relevance_weight: float = 0.7) -> pd.DataFrame:
    primary_term = query.lower().split()[0]
    genre_mask = tracks["track_genre"].str.contains(primary_term, case=False)
    if not genre_mask.any():
        raise ValueError(f"No tracks match genre term in query: {query!r}")

    candidate_indices = tracks.index[genre_mask].to_numpy()
    query_vector = vectorizer.transform([query.lower()])
    similarity = cosine_similarity(query_vector, descriptor_matrix[candidate_indices])[0]
    popularity_norm = tracks.loc[candidate_indices, "popularity"].to_numpy() / 100

    combined_score = relevance_weight * similarity + (1 - relevance_weight) * popularity_norm
    ranked_order = combined_score.argsort()[::-1][:top_n]
    ranked_indices = candidate_indices[ranked_order]

    columns = ["track_id", "track_name", "artists", "track_genre", "mood_label", "popularity"]
    recommendations = tracks.loc[ranked_indices, columns].copy()
    recommendations["combined_score"] = combined_score[ranked_order]
    return recommendations.reset_index(drop=True)

In [5]:
for query in ["acoustic calm", "rock energetic", "jazz"]:
    print(f"\n{query!r}")
    display(recommend_by_genre_query(query, top_n=5))


'acoustic calm'


,track_id,track_name,artists,track_genre,mood_label,popularity,combined_score
0,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,acoustic,calm_dark,82,0.806035
1,4E6cwWJWZw2zWf7VFbH7wf,Love Song,Sara Bareilles,acoustic,bright_energetic,73,0.797361
2,5p9XWUdvbUzmPCukOmwoU3,Suddenly I See,KT Tunstall,acoustic,bright_energetic,73,0.794318
3,63bmIgH9sS6sX5Sc7MetGq,Habang Buhay,Zack Tabudlo,acoustic,intense_dark,69,0.785965
4,0zANeX4R6uWb82gCQAguOD,Habang Buhay,Zack Tabudlo,acoustic,intense_dark,69,0.785965



'rock energetic'


,track_id,track_name,artists,track_genre,mood_label,popularity,combined_score
0,7DbdUf8aHSYoliSjO6LZv6,"Sex, Drugs, Etc.",Beach Weather,rock,intense_dark,90,0.783349
1,2DgdHcjWmO3qd50RzuBLgZ,House of Memories,Panic! At The Disco,rock,intense_dark,85,0.764269
2,2GiJYvgVaD2HtM8GqD9EgQ,Electric Love,BØRNS,rock,intense_dark,82,0.759349
3,0pqnGHJpmpxLKifKRmU6WP,Believer,Imagine Dragons,rock,bright_energetic,88,0.754533
4,0GONea6G2XdnHWjNZd6zt3,Summer Of '69,Bryan Adams,rock,bright_energetic,82,0.754478



'jazz'


,track_id,track_name,artists,track_genre,mood_label,popularity,combined_score
0,1IqF5PUDUnaykHLs0RWbDO,Time Moves Slow,BADBADNOTGOOD;Samuel T. Herring,jazz,calm_dark,71,0.796760
1,5I9sHwLDX28tLtzVgKLtpr,Everybody Loves Somebody,Dean Martin,jazz,calm_dark,68,0.785251
2,1qCQTy0fTXerET4x8VHyr9,What A Wonderful World,Louis Armstrong,jazz,calm_dark,73,0.780165
3,1ko2lVN0vKGUl9zrU0qSlT,Just the Two of Us (feat. Bill Withers),"Grover Washington, Jr.;Bill Withers",jazz,calm_positive,77,0.771293
4,7iaQV6plfQRead7XdleWM4,Aftermath,Caravan Palace,jazz,intense_dark,56,0.768817


## Save

In [6]:
save_npz(MODEL_DIR / "tfidf_matrix.npz", descriptor_matrix)
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")

tfidf_config = {
    "model": "tfidf",
    "use_case": "genre",
    "descriptor_columns": descriptor_columns,
    "relevance_weight": 0.7,
    "candidate_pool": "genre_substring_match",
}
with open(MODEL_DIR / "tfidf_config.json", "w", encoding="utf-8") as file:
    json.dump(tfidf_config, file, indent=2)